# Notebook 02: Data Preprocessing
**Thesis:** Autonomous Threat Hunting: ML-Based MITRE ATT&CK Technique Detection

**What this notebook does:**
- Loads the parsed CSV from Notebook 01
- Cleans and prepares the data for machine learning
- Encodes text columns into numbers (ML only understands numbers)
- Splits data into training set and test set
- Saves the ready-to-use arrays for Notebook 03 (model training)

**Input:** `data/processed/splunk_parsed.csv`  
**Output:** `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`

## Cell 1: Import libraries

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os
import warnings
warnings.filterwarnings('ignore')

print('=== NOTEBOOK 02: DATA PREPROCESSING ===')
print('All libraries imported successfully')

=== NOTEBOOK 02: DATA PREPROCESSING ===
All libraries imported successfully


## Cell 2: Load the parsed CSV from Notebook 01

In [9]:
csv_path = '../data/processed/splunk_parsed.csv'
df = pd.read_csv(csv_path)

print(f'Loaded: {csv_path}')
print(f'Shape: {df.shape}  (rows, columns)')
print()
print('Columns available:')
for col in df.columns:
    print(f'  - {col}')
print()
print('Class distribution:')
print(df['technique'].value_counts())
print()
df.head(3)

Loaded: ../data/processed/splunk_parsed.csv
Shape: (11451, 15)  (rows, columns)

Columns available:
  - technique
  - raw_length
  - EventCode
  - ComputerName
  - AccountName
  - ProcessName
  - CommandLine
  - CommandLineLength
  - LogonType
  - FailureReason
  - IsFailedLogon
  - IsSuccessLogon
  - IsProcessCreation
  - IsPrivilegeUse
  - HasCommandLine

Class distribution:
technique
T1021    10914
T1110      536
T1055        1
Name: count, dtype: int64



,technique,raw_length,EventCode,ComputerName,AccountName,ProcessName,CommandLine,CommandLineLength,LogonType,FailureReason,IsFailedLogon,IsSuccessLogon,IsProcessCreation,IsPrivilegeUse,HasCommandLine
0,T1110,87,0,unknown,unknown,unknown,NaN,0,0,none,0,0,0,0,0
1,T1110,471,4689,win-host-816.attackrange.local,WIN-HOST-816$,C:\Program Files\SplunkUniversalForwarder\bin\...,NaN,0,0,none,0,0,0,0,0
2,T1110,87,0,unknown,unknown,unknown,NaN,0,0,none,0,0,0,0,0


## Cell 3: Inspect data quality

In [10]:
print('=== DATA QUALITY CHECK ===')
print()
print('--- Missing values per column ---')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
quality_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
print(quality_df)
print()
print('--- Data types ---')
print(df.dtypes)

=== DATA QUALITY CHECK ===

--- Missing values per column ---
                   missing_count  missing_%
technique                      0       0.00
raw_length                     0       0.00
EventCode                      0       0.00
ComputerName                   0       0.00
AccountName                    0       0.00
ProcessName                    0       0.00
CommandLine                11318      98.84
CommandLineLength              0       0.00
LogonType                      0       0.00
FailureReason                  0       0.00
IsFailedLogon                  0       0.00
IsSuccessLogon                 0       0.00
IsProcessCreation              0       0.00
IsPrivilegeUse                 0       0.00
HasCommandLine                 0       0.00

--- Data types ---
technique              str
raw_length           int64
EventCode            int64
ComputerName           str
AccountName            str
ProcessName            str
CommandLine            str
CommandLineLength    int6

## Cell 4: Select features and handle missing values

**Features** are the input columns the ML model uses to learn patterns.
**Target** is `technique` — the column we want the model to predict.

We also remove T1055 from training because it has only 1 sample —
too few to learn from or evaluate properly.

In [11]:
print('=== STEP 1: SELECT FEATURES ===')
print()

# These are ALL the real feature columns from our CSV
# (everything except 'technique' which is the target label)
feature_columns = [
    'raw_length',
    'EventCode',
    'ComputerName',
    'AccountName',
    'ProcessName',
    'CommandLine',
    'CommandLineLength',
    'LogonType',
    'FailureReason',
    'IsFailedLogon',
    'IsSuccessLogon',
    'IsProcessCreation',
    'IsPrivilegeUse',
    'HasCommandLine'
]

print(f'Feature columns selected: {len(feature_columns)}')
print(f'Target column: technique')
print()

# Remove T1055 — it has only 1 sample, which is too few to split or evaluate
df_clean = df[df['technique'] != 'T1055_ProcessInjection'].copy()
print('Removed T1055_ProcessInjection (only 1 sample — not enough to train/test)')
print()
print('Remaining class distribution:')
print(df_clean['technique'].value_counts())
print()

# Fill any missing values with 'unknown' for text, 0 for numbers
for col in feature_columns:
    if df_clean[col].dtype == 'object':
        df_clean[col] = df_clean[col].fillna('unknown')
    else:
        df_clean[col] = df_clean[col].fillna(0)

print(f'Final working shape: {df_clean.shape}')
print('Missing values after fill:', df_clean[feature_columns].isnull().sum().sum())

=== STEP 1: SELECT FEATURES ===

Feature columns selected: 14
Target column: technique

Removed T1055_ProcessInjection (only 1 sample — not enough to train/test)

Remaining class distribution:
technique
T1021    10914
T1110      536
T1055        1
Name: count, dtype: int64

Final working shape: (11451, 15)
Missing values after fill: 0


## Cell 5: Encode text columns into numbers

ML models only understand numbers. Text values like `'lsass.exe'`
must be converted to integers like `0`, `1`, `2`, etc.
This is called **Label Encoding**.

In [12]:
print('=== STEP 2: ENCODE TEXT COLUMNS ===')
print()

df_encoded = df_clean.copy()
encoders = {}

# Find which feature columns contain text
text_cols = [col for col in feature_columns if df_encoded[col].dtype == 'object']
num_cols  = [col for col in feature_columns if df_encoded[col].dtype != 'object']

print(f'Text columns to encode: {text_cols}')
print(f'Numeric columns (no change needed): {num_cols}')
print()

for col in text_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    encoders[col] = le
    print(f'  Encoded "{col}": {len(le.classes_)} unique values → 0 to {len(le.classes_)-1}')

# Encode the target label
le_target = LabelEncoder()
df_encoded['technique_encoded'] = le_target.fit_transform(df_encoded['technique'])
print()
print('Target encoding:')
for i, name in enumerate(le_target.classes_):
    print(f'  {i} = {name}')

print()
print('Encoded dataframe (first 3 rows):')
df_encoded[feature_columns].head(3)

=== STEP 2: ENCODE TEXT COLUMNS ===

Text columns to encode: ['CommandLine']
Numeric columns (no change needed): ['raw_length', 'EventCode', 'ComputerName', 'AccountName', 'ProcessName', 'CommandLineLength', 'LogonType', 'FailureReason', 'IsFailedLogon', 'IsSuccessLogon', 'IsProcessCreation', 'IsPrivilegeUse', 'HasCommandLine']

  Encoded "CommandLine": 11 unique values → 0 to 10

Target encoding:
  0 = T1021
  1 = T1055
  2 = T1110

Encoded dataframe (first 3 rows):


,raw_length,EventCode,ComputerName,AccountName,ProcessName,CommandLine,CommandLineLength,LogonType,FailureReason,IsFailedLogon,IsSuccessLogon,IsProcessCreation,IsPrivilegeUse,HasCommandLine
0,87,0,unknown,unknown,unknown,8,0,0,none,0,0,0,0,0
1,471,4689,win-host-816.attackrange.local,WIN-HOST-816$,C:\Program Files\SplunkUniversalForwarder\bin\...,8,0,0,none,0,0,0,0,0
2,87,0,unknown,unknown,unknown,8,0,0,none,0,0,0,0,0


## Cell 6: Split into training set and test set

- **Training set (80%)** — the model learns from this  
- **Test set (20%)** — hidden from the model during training; used to measure performance

In [13]:
print('=== STEP 3: TRAIN / TEST SPLIT ===')
print()

X = df_encoded[feature_columns]
y = df_encoded['technique_encoded']

print(f'X shape (features): {X.shape}')
print(f'y shape (labels):   {y.shape}')
print()

# stratify=y ensures both classes appear proportionally in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'Training set:  {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test set:      {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.1f}%)')
print()
print('Class distribution in training set:')
for code, count in y_train.value_counts().items():
    name = le_target.classes_[code]
    print(f'  {name}: {count} samples')
print()
print('Class distribution in test set:')
for code, count in y_test.value_counts().items():
    name = le_target.classes_[code]
    print(f'  {name}: {count} samples')

=== STEP 3: TRAIN / TEST SPLIT ===

X shape (features): (11451, 14)
y shape (labels):   (11451,)

Training set:  9160 rows (80.0%)
Test set:      2291 rows (20.0%)

Class distribution in training set:
  T1021: 8736 samples
  T1110: 423 samples
  T1055: 1 samples

Class distribution in test set:
  T1021: 2178 samples
  T1110: 113 samples


## Cell 7: Save preprocessed data to disk

In [14]:
print('=== STEP 4: SAVE PREPROCESSED DATA ===')
print()

output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

X_train.to_csv(f'{output_dir}/X_train.csv', index=False)
X_test.to_csv(f'{output_dir}/X_test.csv', index=False)
y_train.to_csv(f'{output_dir}/y_train.csv', index=False)
y_test.to_csv(f'{output_dir}/y_test.csv', index=False)

# Save label map so we can decode predictions in Notebook 03
label_map = pd.DataFrame({
    'encoded': range(len(le_target.classes_)),
    'technique': le_target.classes_
})
label_map.to_csv(f'{output_dir}/label_map.csv', index=False)

print('Files saved to data/processed/:')
print('  X_train.csv   — training features')
print('  X_test.csv    — test features')
print('  y_train.csv   — training labels')
print('  y_test.csv    — test labels')
print('  label_map.csv — maps numbers back to technique names')
print()
print('Label map:')
print(label_map)
print()

=== STEP 4: SAVE PREPROCESSED DATA ===

Files saved to data/processed/:
  X_train.csv   — training features
  X_test.csv    — test features
  y_train.csv   — training labels
  y_test.csv    — test labels
  label_map.csv — maps numbers back to technique names

Label map:
   encoded technique
0        0     T1021
1        1     T1055
2        2     T1110

